In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatOllama(model="llama3.2", temperature=0.3)
agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='64edbc3b-2ca2-4da3-b0cd-7468ca97c8f6'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T07:56:41.016857206Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6717691323, 'load_duration': 6413872236, 'prompt_eval_count': 163, 'prompt_eval_duration': 71013880, 'eval_count': 21, 'eval_duration': 216845769, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09eea-fbd6-7050-9568-9eb5961a5d49-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'cf7bc438-910c-4b8c-b721-4bbfec4a8b82', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 21, 'total_tokens': 184}),
              ToolMessage(content='Successfully updated favourite colour', name='

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'Hello',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='09c62a24-946b-4b2d-b1f2-bfc92f6a2c64'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T07:56:51.460988342Z', 'done': True, 'done_reason': 'stop', 'total_duration': 697759010, 'load_duration': 127032438, 'prompt_eval_count': 164, 'prompt_eval_duration': 317173579, 'eval_count': 21, 'eval_duration': 240037502, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09eeb-38c8-7701-ad1a-9634878b73ce-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'Hello'}, 'id': 'bbbe4e74-573b-467e-81c8-1de95c976dd2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 164, 'output_tokens': 21, 'total_tokens': 185}),
              ToolMessage(content='Successfully updated favourite colour', name='update_fav

## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='6b6c3285-a6fc-4188-b9e5-5f7256d52a92'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T07:57:09.046191415Z', 'done': True, 'done_reason': 'stop', 'total_duration': 719032449, 'load_duration': 134171347, 'prompt_eval_count': 196, 'prompt_eval_duration': 320948514, 'eval_count': 21, 'eval_duration': 232058712, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09eeb-7d65-71f3-96d8-e8f400ed54ff-0', tool_calls=[{'name': 'read_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '9e421973-faca-4ab4-8335-675bae08d5be', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 21, 'total_tokens': 217}),
              ToolMessage(content='No favourite colour found in state', name='read_favourite_colour', id='ec0a17fd

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='6b6c3285-a6fc-4188-b9e5-5f7256d52a92'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T07:57:09.046191415Z', 'done': True, 'done_reason': 'stop', 'total_duration': 719032449, 'load_duration': 134171347, 'prompt_eval_count': 196, 'prompt_eval_duration': 320948514, 'eval_count': 21, 'eval_duration': 232058712, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09eeb-7d65-71f3-96d8-e8f400ed54ff-0', tool_calls=[{'name': 'read_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '9e421973-faca-4ab4-8335-675bae08d5be', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 21, 'total_tokens': 217}),
              ToolMessage(content='No favourite colour found in state', name='read_favourite_colour', id='ec0a17fd